In [ ]:
import asyncio
import httpx
import pandas as pd
import nltk
from tqdm import tqdm

nltk.download('punkt_tab', quiet=True)

True

In [ ]:


GATEWAY_URL = "http://localhost:8000/chat"
K = 5  # número de votos
CONCURRENCY_LIMIT = 20  # evita sobrecarga

semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)


# -------------------------
# 🔥 chamada ao gateway
# -------------------------
async def call_model(client, prompt):
    async with semaphore:
        try:
            resp = await client.post(
                GATEWAY_URL,
                json={"prompt": prompt},
                timeout=60
            )
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except:
            return ""


# -------------------------
# 🔥 avalia uma sentença (K votos em paralelo)
# -------------------------
async def avaliar_sentenca(client, contexto, sent):
    prompt = f"Context: {contexto}\n\nStatement: {sent}\n\nIs the statement faithful to the context?"

    tasks = [
        call_model(client, prompt)
        for _ in range(K)
    ]

    respostas = await asyncio.gather(*tasks)

    votos_unfaithful = 0

    for texto in respostas:
        texto = texto.lower()
        if "unfaithful" in texto or "hallucination" in texto:
            votos_unfaithful += 1

    return votos_unfaithful / K


# -------------------------
# 🔥 processa um relatório inteiro
# -------------------------
async def processar_relatorio(client, row):
    contexto = row['contexto_completo']
    relatorio = row['relatorio_ia']
    sentencas = nltk.sent_tokenize(relatorio)

    tasks = [
        avaliar_sentenca(client, contexto, sent)
        for sent in sentencas
    ]

    scores = await asyncio.gather(*tasks)

    resultados = []
    for sent, score in zip(sentencas, scores):
        resultados.append({
            "id_bo": row['codigo_bo'],
            "sentenca": sent,
            "non_conformity_score": score
        })

    return resultados


# -------------------------
# 🔥 pipeline principal
# -------------------------
async def auditar(csv_relatorios):
    df = pd.read_csv(csv_relatorios)

    print(f"Iniciando auditoria de {len(df)} relatórios...")

    async with httpx.AsyncClient() as client:
        tasks = [
            processar_relatorio(client, row)
            for _, row in df.iterrows()
        ]

        resultados = []
        
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            res = await coro
            resultados.extend(res)

    # salvar CSV final
    df_final = pd.DataFrame(resultados)
    df_final.to_csv("sentencas_auditadas_votos.csv", index=False)

    print("\n✅ Auditoria concluída!")
    print("Arquivo gerado: sentencas_auditadas_votos.csv")

In [ ]:
auditar(csv_path)